# KrishiMitra — PlantVillage Crop Disease Training

This notebook trains a lightweight **MobileNetV2** image classifier and exports a Keras `.keras` model plus the class-name mapping needed by your application.

## Workflow
1. Configure the path to your downloaded PlantVillage dataset.
2. Inspect class folders and image counts.
3. Load training/validation data (and a test set if your dataset includes one).
4. Train MobileNetV2 with transfer learning.
5. Evaluate and export model files to the `models` folder.
6. Test a single leaf image.

**Dataset layout supported:** either one folder per class (for example `Tomato___healthy/`) or a split layout such as `train/`, `valid/` or `validation/`, and `test/`. Extract the downloaded ZIP before running this notebook.

**Note:** PlantVillage is mainly collected under controlled conditions. Good validation performance does not guarantee equal performance on phone photos taken in real fields. Use separate field images for realistic evaluation.

## 1. Install dependencies

In the VS Code terminal, activate your Python environment and run:

```bash
python -m pip install --upgrade pip
pip install -r requirements.txt
```

Then open this notebook and select that environment as the notebook kernel. On a Windows laptop, TensorFlow can train on the CPU. Recent TensorFlow versions do not provide native-Windows NVIDIA GPU support; GPU training typically requires WSL2. See the official [TensorFlow installation guide](https://www.tensorflow.org/install/pip).

In [ ]:
import os
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices("GPU"))

## 2. Set your project and dataset paths

Put your downloaded dataset in the project folder if practical. For example:

```text
crop-disease-model/
├── PlantVillage/                  # extracted dataset (or a parent folder)
├── notebooks/
│   └── crop_disease_training.ipynb
├── models/                         # notebook writes model artifacts here
└── requirements.txt
```

**Edit `DATASET_ROOT` below** so it points to the folder where you extracted PlantVillage. If your folder contains a nested directory like `PlantVillage-Dataset/raw/color`, the notebook tries common layouts automatically. You can always set `CLASS_ROOT` manually if the auto-detection does not find your layout.

In [ ]:
# If the notebook lives in a `notebooks` folder, use its parent as the project directory.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

# CHANGE THIS PATH to the directory you extracted.
# Example Windows path: Path(r"D:\Datasets\PlantVillage")
DATASET_ROOT = PROJECT_DIR / "PlantVillage"

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (160, 160)
BATCH_SIZE = 32
INITIAL_EPOCHS = 12

print("Project directory:", PROJECT_DIR.resolve())
print("Dataset root:", DATASET_ROOT.resolve())
print("Dataset root exists:", DATASET_ROOT.exists())
print("Model output directory:", MODEL_DIR.resolve())

## 3. Detect the dataset layout and count images

The cell below checks common folder layouts. Each disease class should have its own folder containing images. If this cell raises a `FileNotFoundError`, correct `DATASET_ROOT` or manually set `CLASS_ROOT` to the directory directly containing class folders.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}


def has_class_folders(folder: Path, min_classes: int = 2) -> bool:
    """True if folder contains at least min_classes subfolders with direct image files."""
    if not folder.exists() or not folder.is_dir():
        return False
    count = 0
    for subfolder in folder.iterdir():
        if subfolder.is_dir():
            try:
                if any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in subfolder.iterdir()):
                    count += 1
            except PermissionError:
                pass
    return count >= min_classes


# Candidate locations where class folders commonly live.
candidates = [
    DATASET_ROOT,
    DATASET_ROOT / "PlantVillage",
    DATASET_ROOT / "plantvillage",
    DATASET_ROOT / "PlantVillage-Dataset" / "raw" / "color",
    DATASET_ROOT / "raw" / "color",
    DATASET_ROOT / "color",
]

# If a train folder exists and has class subfolders, treat its parent as a split-layout root.
SPLIT_ROOT = None
for candidate in candidates:
    if has_class_folders(candidate / "train"):
        SPLIT_ROOT = candidate
        break

CLASS_ROOT = None
if SPLIT_ROOT is None:
    for candidate in candidates:
        if has_class_folders(candidate):
            CLASS_ROOT = candidate
            break
else:
    CLASS_ROOT = SPLIT_ROOT / "train"

if CLASS_ROOT is None:
    raise FileNotFoundError(
        "Could not find class folders automatically. Set DATASET_ROOT to the extracted dataset folder, "
        "or set CLASS_ROOT manually to the folder directly containing class folders."
    )

print("Detected class folder root:", CLASS_ROOT.resolve())
print("Split-layout root:", SPLIT_ROOT.resolve() if SPLIT_ROOT else "None (will split class folders into train/validation)")

class_dirs = sorted([p for p in CLASS_ROOT.iterdir() if p.is_dir()])
class_counts = {}
for folder in class_dirs:
    class_counts[folder.name] = sum(
        1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

print("Number of classes:", len(class_counts))
print("Total images:", sum(class_counts.values()))
for name, count in class_counts.items():
    print(f"{name}: {count}")

if len(class_counts) < 2:
    raise ValueError("Found fewer than two classes. Verify that CLASS_ROOT points directly to the disease folders.")

## 4. Load training, validation, and optional test sets

- If `train/` plus `valid/` or `validation/` exists, those are used directly.
- If `train/` and `test/` exist without validation, 20% of training data is used for validation.
- If the dataset only has class folders, it is split into 80% training and 20% validation. In that case, there is no independent test set.

The class order is saved later to `models/class_names.json`; keep that file alongside the trained model.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

if SPLIT_ROOT is not None:
    TRAIN_DIR = SPLIT_ROOT / "train"
    TEST_DIR = SPLIT_ROOT / "test" if has_class_folders(SPLIT_ROOT / "test") else None
    VAL_DIR = None
    for val_name in ("valid", "validation", "val"):
        possible = SPLIT_ROOT / val_name
        if has_class_folders(possible):
            VAL_DIR = possible
            break
else:
    TRAIN_DIR = CLASS_ROOT
    TEST_DIR = None
    VAL_DIR = None

if VAL_DIR is not None:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        label_mode="int", seed=SEED, shuffle=True
    )
    class_names = train_ds.class_names
    val_ds = tf.keras.utils.image_dataset_from_directory(
        VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        label_mode="int", class_names=class_names, shuffle=False
    )
else:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, validation_split=0.20, subset="training", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int", shuffle=True
    )
    class_names = train_ds.class_names
    val_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR, validation_split=0.20, subset="validation", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int",
        class_names=class_names, shuffle=False
    )

num_classes = len(class_names)
if len(class_names) != len(class_counts):
    print("Warning: detected class count and loaded class count differ. Recheck your folder layout.")

if TEST_DIR is not None:
    test_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        label_mode="int", class_names=class_names, shuffle=False
    )
else:
    test_ds = None

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
if test_ds is not None:
    test_ds = test_ds.prefetch(AUTOTUNE)

print("Classes:", num_classes)
print(class_names)
print("Independent test set available:", test_ds is not None)

with open(MODEL_DIR / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2, ensure_ascii=False)
print("Saved class mapping to:", MODEL_DIR / "class_names.json")

## 5. Visualize sample images

In [ ]:
images, labels = next(iter(train_ds))
plt.figure(figsize=(12, 9))
for i in range(min(9, len(images))):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[int(labels[i])], fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Build a lightweight MobileNetV2 model

The model accepts RGB images resized to `160 × 160`. Its rescaling layer is included inside the saved model, so your inference API should pass pixel values in the normal `0–255` range and **must not normalize them a second time**. The first stage freezes the pretrained ImageNet backbone and trains the classifier head.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="data_augmentation")

base_model = keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="leaf_image")
x = data_augmentation(inputs)
x = layers.Rescaling(1.0 / 127.5, offset=-1, name="mobilenetv2_rescaling")(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)

model = keras.Model(inputs, outputs, name="krishimitra_mobilenetv2")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 7. Train and save the best checkpoint

The best checkpoint is selected by validation accuracy. Early stopping reduces unnecessary training when validation loss stops improving. The first run downloads MobileNetV2 ImageNet weights, so an internet connection may be needed once.

In [ ]:
BEST_MODEL_PATH = MODEL_DIR / "best_crop_model.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH), monitor="val_accuracy",
        mode="max", save_best_only=True, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=2,
        min_lr=1e-6, verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=callbacks
)

## 8. Plot training history

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Optional fine-tuning

Skip this section on the first run if you want a simpler training pipeline. Fine-tuning can help, but it does not always improve validation performance. The cell loads the best first-stage model, unfreezes only the last 30 backbone layers (while keeping BatchNormalization layers frozen), trains with a very small learning rate, and saves a fine-tuned checkpoint only when validation accuracy improves.

In [ ]:
# Optional: run this cell only after the first training stage has completed.
# Reload the best first-stage checkpoint before fine-tuning.
finetune_model = keras.models.load_model(BEST_MODEL_PATH)
backbone = next(layer for layer in finetune_model.layers if isinstance(layer, keras.Model) and "mobilenetv2" in layer.name.lower())
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

finetune_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

FINE_TUNED_PATH = MODEL_DIR / "best_crop_model_finetuned.keras"
finetune_callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(FINE_TUNED_PATH), monitor="val_accuracy",
        mode="max", save_best_only=True, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True, verbose=1
    )
]

fine_tune_history = finetune_model.fit(
    train_ds, validation_data=val_ds, epochs=5,
    callbacks=finetune_callbacks
)
print("Fine-tuned checkpoint path:", FINE_TUNED_PATH)

## 10. Evaluate the best saved model and export the deployment model

If you ran fine-tuning, the notebook uses the fine-tuned checkpoint only if it exists. Otherwise, it uses the best first-stage checkpoint. It evaluates on the independent test folder when available; otherwise it reports validation performance (which was also used during model selection).

In [ ]:
FINE_TUNED_PATH = MODEL_DIR / "best_crop_model_finetuned.keras"
if FINE_TUNED_PATH.exists():
    best_model = keras.models.load_model(FINE_TUNED_PATH)
    print("Using fine-tuned checkpoint.")
else:
    best_model = keras.models.load_model(BEST_MODEL_PATH)
    print("Using first-stage best checkpoint.")

final_eval_ds = test_ds if test_ds is not None else val_ds
final_eval_name = "test" if test_ds is not None else "validation"
loss, accuracy = best_model.evaluate(final_eval_ds, verbose=1)
print(f"{final_eval_name.title()} loss: {loss:.4f}")
print(f"{final_eval_name.title()} accuracy: {accuracy * 100:.2f}%")

FINAL_MODEL_PATH = MODEL_DIR / "crop_disease_mobilenetv2.keras"
best_model.save(FINAL_MODEL_PATH)
print("Exported model:", FINAL_MODEL_PATH.resolve())
print("Class labels:", (MODEL_DIR / "class_names.json").resolve())

## 11. Per-class evaluation report

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in final_eval_ds], axis=0)
y_prob = best_model.predict(final_eval_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
fig, ax = plt.subplots(figsize=(max(10, len(class_names) * 0.45), max(8, len(class_names) * 0.4)))
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
display.plot(ax=ax, xticks_rotation=90, colorbar=False, values_format="d")
plt.title(f"Confusion matrix ({final_eval_name} set)")
plt.tight_layout()
plt.show()

## 12. Predict one new leaf image

Set `IMAGE_PATH` to a real image file that was not used for training. This is a quick smoke test, not a reliable estimate of real-world field performance.

In [ ]:
IMAGE_PATH = Path(r"D:\path\to\your_leaf_image.jpg")  # CHANGE THIS

if not IMAGE_PATH.exists():
    print("Update IMAGE_PATH to an existing leaf photo, then rerun this cell.")
else:
    image = Image.open(IMAGE_PATH).convert("RGB").resize(IMG_SIZE)
    image_array = np.asarray(image, dtype=np.float32)[None, ...]
    probabilities = best_model.predict(image_array, verbose=0)[0]
    top_indices = np.argsort(probabilities)[::-1][:5]

    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

    print("Top predictions:")
    for idx in top_indices:
        print(f"{class_names[int(idx)]}: {probabilities[idx]:.2%}")

## 13. Files to copy into KrishiMitra

After the notebook finishes, you will have:

```text
models/
├── crop_disease_mobilenetv2.keras   # deploy this model
├── class_names.json                 # deploy with the model
├── best_crop_model.keras            # training checkpoint
└── best_crop_model_finetuned.keras  # optional checkpoint, if fine-tuning was run
```

For normal prediction, the essential files are **`crop_disease_mobilenetv2.keras` and `class_names.json`**. Load the model once when your Python API starts. Resize input images to `160 × 160` and feed RGB pixel values in the `0–255` range because rescaling is already inside the model. Use the same class-name JSON to convert prediction indices to labels.

**Integration note:** this model predicts only among the PlantVillage classes included in training. Consider returning an “uncertain / please retake image” result for low-confidence predictions, and validate against real field images before relying on predictions in an agriculture advisory workflow.